In [1]:
!pip install sentence-transformers
!pip install spacy
!pip install nltk
!pip install streamlit

!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 92.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 77.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy
import nltk
import string
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

from google.colab import files

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
uploaded = files.upload()

Saving freedom_fighters_docs.txt to freedom_fighters_docs.txt


In [4]:
with open("freedom_fighters_docs.txt", "r", encoding="utf-8") as f:
    documents = f.read()

print(documents[:500])

Mahatma Gandhi was born on 2 October 1869 in Porbandar, Gujarat.
Mahatma Gandhi was a leader of the Indian independence movement against British rule.
Mahatma Gandhi followed the principles of nonviolence and truth.
Mahatma Gandhi led important movements such as the Non-Cooperation Movement.
Mahatma Gandhi led the Quit India Movement in 1942.

Bhagat Singh was an Indian revolutionary freedom fighter.
Bhagat Singh played an important role in the struggle against British rule.
Bhagat Singh was bor


In [5]:
sentences = sent_tokenize(documents)

print("Total sentences:", len(sentences))
print(sentences[0])

Total sentences: 91
Mahatma Gandhi was born on 2 October 1869 in Porbandar, Gujarat.


In [6]:
nlp = spacy.load("en_core_web_sm")

In [7]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
sentence_embeddings = model.encode(sentences)

In [9]:
def extract_entities(text):

    doc = nlp(text)

    entities = [(ent.text, ent.label_) for ent in doc.ents]

    return entities

In [10]:
def get_answer(question):

    entities = extract_entities(question)

    question_embedding = model.encode([question])

    similarity = cosine_similarity(question_embedding, sentence_embeddings)

    best_index = similarity.argmax()

    answer = sentences[best_index]

    return answer, entities

In [11]:
while True:

    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        break

    answer, entities = get_answer(question)

    print("Entities:", entities)
    print("Answer:", answer)


Ask a question (type 'exit' to stop): when was jawaharlal nehru bron
Entities: []
Answer: Jawaharlal Nehru was an important leader of the Indian National Congress.

Ask a question (type 'exit' to stop): when was jawaharlal nehru born
Entities: []
Answer: Jawaharlal Nehru was born on 14 November 1889 in Allahabad.

Ask a question (type 'exit' to stop): exit
